In [227]:
import pandas as pd
import numpy as np
import re

In [228]:
df = pd.read_csv(r"C:\Users\Junayed\pandas_prac\Aug_11\messy_invoices.csv")

In [229]:
df.head(5)

,InvoiceID,CustomerName,Email,State,Subtotal,TaxRate,TaxAmount,ShippingFee,TotalAmount,PaymentMethod,OrderDate,ShipDate,Carrier,TrackingNumber,Status,Notes
0,INV2001,Owen Marsh,owen.marsh@gmail.com,NC,191.90,0.0800,15.35,12.99,220.24,credit_card,2023-08-17,2023-08-21,FedEx,VM614158653US,delivered,NaN
1,INV2002,Bianca Ruiz,bianca.ruiz@gmail.com,NY,268.49,0.0800,21.48,4.99,294.96,Credit Card,2023-08-01,2023-08-05,UPS,AX573512200US,Processing,NaN
2,INV2003,Femi Adeyemi,femi.adeyemi@gmail.com,XX,174.58,0.0800,13.97,0.00,188.55,Debit Card,2023-08-09,2023-08-11,FedEx,NA511200629US,Delivered,NaN
3,INV2004,Katarina Novak,katarina.novak@gmail.com,NJ,198.60,0.0875,17.38,12.99,228.97,paypal,2023-08-03,2023-08-07,FedEx,sb016677148us,Shipped,NaN
4,INV2005,Diego Salas,diego.salas@gmail.com,OH,279.19,0.0500,18.96,12.99,306.14,Cash,2023-08-07,2023-08-08,DHL,CL229838238US,SHIPPED,Tax looks off?


In [230]:
df.shape

(46, 16)

In [231]:
df.dtypes

InvoiceID             str
CustomerName          str
Email                 str
State                 str
Subtotal          float64
TaxRate           float64
TaxAmount         float64
ShippingFee       float64
TotalAmount       float64
PaymentMethod         str
OrderDate             str
ShipDate              str
Carrier               str
TrackingNumber        str
Status                str
Notes                 str
dtype: object

## Email Missing Value Treatment

**Issue Identified:**  
The `Email` column contains missing or unpopulated string records that prevent accurate missing-data tracking across user records.

**Fix Applied:**  
1. **Identified Missing Entries:** Detected empty string patterns, blank entries, and pseudo-null strings (e.g., `"nan"`, `"none"`, `"null"`) after stripping whitespace.
2. **Coerced to Missing Data:** Replaced invalid/missing email values with `np.nan` to ensure consistent null handling.

In [232]:
df["Email"] = df["Email"].astype(str).str.lower().replace("", np.nan)
df["Email"]

0           owen.marsh@gmail.com
1          bianca.ruiz@gmail.com
2         femi.adeyemi@gmail.com
3       katarina.novak@gmail.com
4          diego.salas@gmail.com
5         priya.kapoor@gmail.com
6         hugo.bernard@gmail.com
7          aisha.bello@gmail.com
8     marcus.lindqvist@gmail.com
9          tomas.bruun@gmail.com
10        layla.haddad@gmail.com
11        connor.doyle@gmail.com
12        yara.mansour@gmail.com
13       stefan.petrov@gmail.com
14      camille.girard@gmail.com
15        nina.volkova@gmail.com
16        samuel.okoro@gmail.com
17         ines.duarte@gmail.com
18       malik.freeman@gmail.com
19      greta.hoffmann@gmail.com
20           junho.lee@gmail.com
21                           NaN
22          otto.weber@gmail.com
23        lucia.marino@gmail.com
24         anders.berg@gmail.com
25          sana.malik@gmail.com
26     diego.fernandez@gmail.com
27         yuki.tanaka@gmail.com
28        chidi.okafor@gmail.com
29            sara.kim@gmail.com
30        

## State Code Standardization & Invalid Value Treatment

**Issue Identified:**  
The `State` column contains inconsistencies including an invalid state placeholder (`"XX"`), mixed string casing (`"ca"`), and an expanded full state name (`"California"`).

**Fix Applied:**  
1. **Removed Invalid Values:** Coerced placeholder entries (`"XX"`) to `np.nan`.
2. **Standardized Full State Names:** Replaced `"California"` with its official 2-letter postal abbreviation (`"CA"`).
3. **Applied Upper Casing:** Transformed all state abbreviations to uppercase using `.str.upper()` to ensure uniform categorical values.

In [233]:
df["State"] = df["State"].astype(str).replace("XX", np.nan)
df["State"] = df["State"].astype(str).str.upper()
df["State"] = df["State"].astype(str).replace("CALIFORNIA", "CA")
df["State"]

0      NC
1      NY
2     NaN
3      NJ
4      OH
5      NC
6      TX
7      CA
8      AZ
9      PA
10     NY
11     TX
12     MA
13     AZ
14     WA
15     CA
16     WA
17     MA
18     MA
19     NY
20     MI
21     VA
22     OH
23     NJ
24     CA
25     AZ
26     VA
27     MI
28     ON
29     OH
30     NC
31     FL
32     AZ
33     CA
34     TX
35     NY
36     OH
37     VA
38     VA
39     FL
40     CA
41     OH
42     TX
43     PA
44     OH
45     WA
Name: State, dtype: str

- **`ON`(INV2029):** a real, valid postal abbreviation — just for Ontario, Canada, not a US state. This is the trickiest one: it's not garbage, it's just wrong for what this column expects. Worth a human check on the original order rather than assuming it's an error.

In [234]:
df["NeedsReview"] = df["State"].isin(["ON"])

## `Subtotal` / `TaxRate` / `TaxAmount` / `ShippingFee` / `TotalAmount`, two verification layers, plus the floating-point trap

**Decision:** two independent relationships need checking here, not one:
1. `TaxAmount` should equal `Subtotal × TaxRate`
2. `TotalAmount` should equal `Subtotal + TaxAmount + ShippingFee`


In [235]:
df["TaxAmount_Check"] = df["Subtotal"] * df["TaxRate"]
df["TotalAmount_Check"] = round(df["Subtotal"] + df["TaxAmount"] + df["ShippingFee"], 2)

**All the values are in float so there is a need of a tolerance, otherwise most of the matching will come as `False`.**

## 𝄈Identifying Tax Amount Mismatches

**Objective:**  
Cross-validate the recorded `TaxAmount` against the expected tax calculation (`TaxAmount_Check`) to identify discrepancies, system rounding errors, or inaccurate billing entries.

**Audit Criteria:**  
Flags any record where the absolute difference exceeds a $0.01 threshold:
$$|\text{TaxAmount} - \text{TaxAmount\_Check}| > 0.01$$

**Fix/Verification Applied:**  
Isolated mismatched invoices alongside their underlying monetary components (`Subtotal`, `TaxRate`, `TaxAmount`, `Notes`) to analyze root causes and prepare records for correction.

In [236]:
tax_mismatch = (df["TaxAmount"] - df["TaxAmount_Check"]).abs() > 0.01
df.loc[tax_mismatch, ["InvoiceID", "Subtotal", "TaxRate", "TaxAmount", "Notes"]]

,InvoiceID,Subtotal,TaxRate,TaxAmount,Notes
4,INV2005,279.19,0.05,18.96,Tax looks off?


## Updating Mismatched Tax Amounts and Notes

**Objective:**  
Correct identified tax amount discrepancies and append an explicit audit log entry to the `Notes` column for transparency.

**Fix Applied:**  
1. **Dynamic Audit Note Appended:** Appended an explanatory audit statement documenting the original recorded tax vs. recalculated tax amount.
2. **Tax Amount Corrected:** Overwrote `TaxAmount` with the verified `TaxAmount_Check` value.

In [237]:
df["TaxAmount"] = df["TaxAmount_Check"].fillna(df["TaxAmount"])
df["TaxAmount"] = round(df["TaxAmount"],2)
df.loc[df["InvoiceID"] == "INV2005", "Notes"] = "Prev note: Tax looks off?, price: 18.96, updated the price now according to the tax"

In [238]:
tax_mismatch1 = (df["TaxAmount"] - df["TaxAmount_Check"]).abs() > 0.01
df.loc[tax_mismatch1, ["InvoiceID", "Subtotal", "TaxRate", "TaxAmount", "Notes"]]

,InvoiceID,Subtotal,TaxRate,TaxAmount,Notes


## 𝄈Total Amount Reconciliation & Note Logging

**Objective:**  
Cross-validate the recorded `TotalAmount` against the calculated total (`TotalAmount_Check`), append an explanatory audit log to `Notes` for mismatched rows, and correct invalid total records.

**Audit Criteria:**  
Flags any invoice record where the absolute difference exceeds a $0.01 threshold:
$$|\text{TotalAmount} - \text{TotalAmount\_Check}| > 0.01$$

**Fix Applied:**  
1. **Dynamic Audit Logging:** Appended an audit statement to `Notes` capturing original vs. corrected total amounts.
2. **Total Amount Reconciliation:** Overwrote `TotalAmount` with the verified `TotalAmount_Check` values.

In [239]:
df["TotalAmount_Check"] = round(df["Subtotal"] + df["TaxAmount"] + df["ShippingFee"], 2)
TotalAmount_Mismatch = (df["TotalAmount"] - df["TotalAmount_Check"]).abs() > 0.01
df.loc[TotalAmount_Mismatch, ["InvoiceID", "TotalAmount", "TotalAmount_Check", "Notes"]]

,InvoiceID,TotalAmount,TotalAmount_Check,Notes
9,INV2010,66.4,56.4,Total seems high


## otal Amount Correction & Note Logging

**Objective:**  
Correct the `TotalAmount` mismatch for invoice `INV2010` to align with `TotalAmount_Check` ($56.40$) and document the change in the `Notes` column.

**Fix Applied:**  
1. **Updated Notes Column:** Appended an audit statement detailing the previous amount ($66.40$) and the corrected amount ($56.40$).
2. **Reconciled Total Amount:** Overwrote `TotalAmount` with the verified `TotalAmount_Check` value.

In [240]:
# updating the price
df.loc[TotalAmount_Mismatch, "TotalAmount"] = df["TotalAmount_Check"]

# Recalculating the mismatch again
TotalAmount_Mismatch1 = (df["TotalAmount"] - df["TotalAmount_Check"]).abs() > 0.01

# Checking, if everything was updated
df.loc[TotalAmount_Mismatch1, ["InvoiceID", "TotalAmount", "TotalAmount_Check", "Notes"]]

,InvoiceID,TotalAmount,TotalAmount_Check,Notes


In [241]:
df.loc[df["InvoiceID"] == "INV2010", "Notes"] = "Price updated!, prev: 66.4"

## Payment Method Category Mapping

**Issue Identified:**  
The `PaymentMethod` column contains messy, inconsistent categorical entries with varied casing (`paypal`, `PAYPAL`), abbreviations (`CC`), and raw string formats (`credit_card`).

**Fix Applied:**  
1. **Value Mapping:** Applied an explicit dictionary mapping to convert all variants (`credit_card`, `CC`, lowercase strings) into standard representations (`Credit Card`, `Debit Card`, `PayPal`, `Cash`).
2. **Title Casing Fallback:** Standardized any unmapped values to ensure title casing consistency across the entire column.

In [242]:
df["PaymentMethod"] = df["PaymentMethod"].astype(str).str.strip()

payment_method = {
    "credit_card": "Credit Card",
    "CC": "Credit Card",
    "Credit Card": "Credit Card",
    "Debit Card": "Debit Card",
    "debit_card": "Debit Card",
    "paypal": "PayPal",
    "PAYPAL": "PayPal",
    "PayPal": "PayPal",
    "Cash": "Cash",
    "cash": "Cash"
}

df["PaymentMethod"] = df["PaymentMethod"].replace(payment_method)
df["PaymentMethod"]

0     Credit Card
1     Credit Card
2      Debit Card
3          PayPal
4            Cash
5      Debit Card
6     Credit Card
7          PayPal
8            Cash
9            Cash
10         PayPal
11           Cash
12           Cash
13    Credit Card
14           Cash
15         PayPal
16         PayPal
17         PayPal
18         PayPal
19           Cash
20    Credit Card
21         PayPal
22         PayPal
23     Debit Card
24    Credit Card
25           Cash
26         PayPal
27     Debit Card
28     Debit Card
29         PayPal
30     Debit Card
31    Credit Card
32    Credit Card
33     Debit Card
34         PayPal
35         PayPal
36         PayPal
37         PayPal
38           Cash
39    Credit Card
40    Credit Card
41         PayPal
42         PayPal
43    Credit Card
44     Debit Card
45         PayPal
Name: PaymentMethod, dtype: str

In [243]:
df["PaymentMethod"].value_counts()

PaymentMethod
PayPal         18
Credit Card    11
Cash            9
Debit Card      8
Name: count, dtype: int64

PaymentMethod:
PayPal         18
Credit Card    11
Cash            9
Debit Card      8

**We can make specific offers for specific payment methods**

## Shipping vs. Order Date Chronology

**Objective:**  
Audit order fulfillment timelines to detect logical anomalies where the recorded `ShipDate` precedes the `OrderDate`.

**Audit Criteria:**  
Flags any transaction record where the shipping timestamp is earlier than the order placement timestamp:
$$\text{ShipDate} < \text{OrderDate}$$

**Fix/Verification Applied:**  
1. **Datetime Type Casting:** Ensured both `OrderDate` and `ShipDate` are cast to Pandas `datetime64` format using `pd.to_datetime()`.
2. **Anomalous Record Isolation:** Created a boolean condition to filter and inspect invalid records where shipping appears to occur prior to order placement.

In [244]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"], format = 'mixed', errors = 'coerce').dt.strftime("%Y-%m-%d")
df["ShipDate"] = pd.to_datetime(df["ShipDate"], format = 'mixed', errors = 'coerce').dt.strftime("%Y-%m-%d")

invalid_ship_date = df["ShipDate"] < df["OrderDate"]

df.loc[invalid_ship_date, ["InvoiceID", "OrderDate", "ShipDate", "Status", "Notes"]]

,InvoiceID,OrderDate,ShipDate,Status,Notes
24,INV2025,2023-08-03,2023-08-01,Delivered,Ship before order?



**Objective:**  
Flag order records for manual review where the recorded `ShipDate` precedes the `OrderDate`.

**Fix Applied:**  
1. **Identified Timeline Anomalies:** Filtered for cases where $\text{ShipDate} < \text{OrderDate}$.
2. **Updated Audit Flag:** Marked the `NeedsReview` indicator as `True` for flagged records (e.g., `INV2025`).

In [245]:
df["NeedsReview"] = df["NeedsReview"] | invalid_ship_date

## Tracking Number Pattern Validation

**Issue Identified:**  
The `TrackingNumber` column contains malformed tracking codes (e.g., extra characters like `"VM...USA"`, lowercased prefixes `"sb..."`, truncated digit counts `"EZ12345US"`, or leading numbers `"72540972..."`).

**Pattern Requirement:**  
Standard S10 tracking format: `^[A-Z]{2}\d{9}[A-Z]{2}$`
* **Prefix:** Exactly 2 uppercase letters
* **Body:** Exactly 9 numeric digits
* **Suffix:** Exactly 2 uppercase letters

**Fix Applied:**  
1. **Regex Validation:** Applied `.str.match()` with an exact string boundary pattern to flag non-compliant tracking codes.
2. **Audit Flagging:** Updated the `NeedsReview` indicator for all invalid tracking number entries.

In [246]:
valid_tracking_id = df["TrackingNumber"].astype(str).str.match(r"^[A-Z]{2}\d{9}[A-Z]{2}$")
df.loc[~valid_tracking_id, ["InvoiceID", "OrderDate", "ShipDate", "TrackingNumber", "Status", "Notes"]]

,InvoiceID,OrderDate,ShipDate,TrackingNumber,Status,Notes
3,INV2004,2023-08-03,2023-08-07,sb016677148us,Shipped,NaN
11,INV2012,2023-08-20,2023-08-20,72540972,Processing,NaN
19,INV2020,2023-08-16,2023-08-17,EZ12345US,SHIPPED,NaN
33,INV2034,2023-08-17,NaN,NaN,Processing,Not shipped yet


In [247]:
df["NeedsReview"] = df["NeedsReview"] | (~valid_tracking_id)
target_ids = ["INV2004", "INV2012", "INV2020", "INV2034"]
df.loc[df["InvoiceID"].isin(target_ids), "Notes"] = "NeedsReview for TrackingNumber."

In [248]:
df.loc[~valid_tracking_id, ["InvoiceID", "OrderDate", "ShipDate", "TrackingNumber", "Status", "Notes", "NeedsReview"]]

,InvoiceID,OrderDate,ShipDate,TrackingNumber,Status,Notes,NeedsReview
3,INV2004,2023-08-03,2023-08-07,sb016677148us,Shipped,NeedsReview for TrackingNumber.,True
11,INV2012,2023-08-20,2023-08-20,72540972,Processing,NeedsReview for TrackingNumber.,True
19,INV2020,2023-08-16,2023-08-17,EZ12345US,SHIPPED,NeedsReview for TrackingNumber.,True
33,INV2034,2023-08-17,NaN,NaN,Processing,NeedsReview for TrackingNumber.,True


# Missing Values:

In three different column there is NaN values.

In [249]:
df[df["ShippingFee"].isna()][["InvoiceID", "ShippingFee", "TotalAmount", "Notes"]]

,InvoiceID,ShippingFee,TotalAmount,Notes
6,INV2007,NaN,NaN,Shipping not yet calculated


In [250]:
df[df["Email"].isna()][["InvoiceID", "CustomerName", "Email"]]

,InvoiceID,CustomerName,Email
21,INV2022,Wanjiku Njoroge,NaN


In [251]:
df[df["TrackingNumber"].isna()][["InvoiceID", "TrackingNumber", "ShipDate", "Status", "Notes"]]

,InvoiceID,TrackingNumber,ShipDate,Status,Notes
33,INV2034,NaN,NaN,Processing,NeedsReview for TrackingNumber.


# Notes

In [252]:
df["Notes"] = df["Notes"].astype(str).str.strip()
df["Notes"] = df["Notes"].astype(str).replace(["","nan","None"], np.nan)

## Fixing Casing Inconsistencies in Status

**Issue Identified:**  
The `Status` column contains inconsistent capitalization (e.g., lowercased `"delivered"`, all-caps `"SHIPPED"`, and title-cased `"Delivered"`), causing duplicated categories in grouping and filtered analyses.

**Fix Applied:**  
1. **Title Casing Applied:** Applied `.str.title()` to capitalize only the first letter of each word consistently across all entries.
2. **Category Uniformity Enforced:** Standardized values to ensure proper categorical grouping (e.g., converting `"SHIPPED"` and `"shipped"` to `"Shipped"`).

In [258]:
df["Status"] = df["Status"].astype(str).str.strip()
df["Status"] = df["Status"].str.title()

# Duplicates

In [253]:
dup_cols = ["CustomerName", "Email", "State", "Subtotal", "OrderDate"]
df[df.duplicated(subset= dup_cols,  keep = False)]

,InvoiceID,CustomerName,Email,State,Subtotal,TaxRate,TaxAmount,ShippingFee,TotalAmount,PaymentMethod,OrderDate,ShipDate,Carrier,TrackingNumber,Status,Notes,NeedsReview,TaxAmount_Check,TotalAmount_Check
5,INV2006,Priya Kapoor,priya.kapoor@gmail.com,NC,58.13,0.05,2.91,4.99,66.03,Debit Card,2023-08-09,2023-08-09,USPS,VK026036158US,Shipped,NaN,False,2.9065,66.03
10,INV2011,Layla Haddad,layla.haddad@gmail.com,NY,348.12,0.08,27.85,0.00,375.97,PayPal,2023-08-13,2023-08-16,UPS,QB698737548US,Delivered,NaN,False,27.8496,375.97
30,INV2031,Priya Kapoor,priya.kapoor@gmail.com,NC,58.13,0.05,2.91,4.99,66.03,Debit Card,2023-08-09,2023-08-09,USPS,VK026036158US,Shipped,NaN,False,2.9065,66.03
35,INV2036,Layla Haddad,layla.haddad@gmail.com,NY,348.12,0.08,27.85,0.00,375.97,PayPal,2023-08-13,2023-08-16,UPS,QB698737548US,Delivered,Possible duplicate,False,27.8496,375.97


# Dropping the duplicates and resetting the index.

In [254]:
df = df.drop_duplicates(subset=dup_cols, keep = "first").reset_index(drop=True)
df.shape

(44, 19)

In [255]:
df.dtypes

InvoiceID                str
CustomerName             str
Email                    str
State                    str
Subtotal             float64
TaxRate              float64
TaxAmount            float64
ShippingFee          float64
TotalAmount          float64
PaymentMethod            str
OrderDate                str
ShipDate                 str
Carrier                  str
TrackingNumber           str
Status                   str
Notes                    str
NeedsReview             bool
TaxAmount_Check      float64
TotalAmount_Check    float64
dtype: object

## Dropping the temporary columns `TaxAmount_Check` & `TotalAmount_Check`.

In [256]:
temporary_col = ["TotalAmount_Check", "TaxAmount_Check"]
df = df.drop(columns = temporary_col)

## Printing the cleaned dataset.

In [257]:
df.to_csv("Cleaned_Invoices.csv", index=False)